In [ ]:
# ============================================================================
# CELL 1: Install dependencies
# ============================================================================
#@title Install dependencies
print("Installing RDKit and PostgreSQL drivers...")
!pip install -q rdkit pandas psycopg2-binary ipywidgets
print("✓ Installation complete!")


In [ ]:

# ============================================================================
# CELL 2: Database engine for the enumerated library
# ============================================================================
#@title Load DB engine

import psycopg2
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw

# Lab → reaction codes shown in the GUI. Update this when you add reactions.
LAB_REACTIONS = {
    "Rousseaux":  ["a1"],
    "Le":         ["a2"],
    "Beauchemin": ["a3", "a4"],
    "Lundgren":   ["a5", "a6", "a7", "a8"],
    "Batey":      ["c1", "c2", "c4"],
    "Wood":       ["c5"],
    "West":       ["c6", "c8"],
}

# Pretty labels for each reaction (shown next to the toggles).
REACTION_LABELS = {
    "a1": "Pyrrolidines",
    "a2": "Carbamoyl fluorides",
    "a3": "Pyridazinones (alkyne)",
    "a4": "Pyridazinones (bromoketone)",
    "a5": "Allylic substitution",
    "a6": "Cyclohexadiene formation",
    "a7": "Diene synthesis",
    "a8": "1,4-additions",
    "c1": "Imides",
    "c2": "Aminothiatriazoles",
    "c4": "Aminotetrazoles",
    "c5": "Truce-Smiles",
    "c6": "[2+2]-cycloaddition",
    "c8": "[4+2]-cycloaddition",
}


class EnumLibraryDB:
    def __init__(self):
        self.conn_params = {
            "host":     "206.12.98.193",
            "database": "pccl_db2",
            "user":     "pccl_guest2",
            "password": "guest_readonly_2026",
        }

    # ------------------------------------------------------------------
    def search(self, *, labs, reactions, mw, hba, hbd, logp, rotb, psa,
               fsp3, qed, exclude_pains, exclude_brenk, exclude_nih,
               in_zinc, query_smiles=None, sim_thresh=0.0, limit=100):
        """
        Filter-then-rank query.  Property filters are applied first (cheap,
        index-backed); similarity search runs only over the surviving rows.
        """
        where = []
        params = []

        if labs:
            if len(labs) == 1:
                where.append("lab = %s")
                params.append(labs[0])
            else:
                where.append("lab = ANY(%s)")
                params.append(list(labs))
        if reactions:
            # Use `= %s` for the single-reaction case so the planner picks
            # idx_enum_reaction_qed (multi-column).  With ANY('{a1}') Postgres
            # falls back to a sequential scan even with the index present.
            if len(reactions) == 1:
                where.append("reaction = %s")
                params.append(reactions[0])
            else:
                where.append("reaction = ANY(%s)")
                params.append(list(reactions))

        # Numeric range filters
        for col, (lo, hi) in [
            ("mw", mw), ("hba", hba), ("hbd", hbd), ("logp", logp),
            ("rotb", rotb), ("psa", psa), ("fsp3", fsp3), ("qed", qed),
        ]:
            where.append(f"{col} BETWEEN %s AND %s")
            params.extend([lo, hi])

        # Alert filters (the columns store an integer: 0 = clean, >0 = alert)
        if exclude_pains: where.append("(pains IS NULL OR pains = 0)")
        if exclude_brenk: where.append("(brenk IS NULL OR brenk = 0)")
        if exclude_nih:   where.append("(nih   IS NULL OR nih   = 0)")
        if in_zinc:       where.append("zinc = 1")

        where_clause = " AND ".join(where) if where else "TRUE"

        if query_smiles:
            # Filter then rank by Tanimoto similarity. The KNN operator <%>
            # uses the GiST index when present.
            sql = f"""
                SELECT smiles, code, reaction, lab, mw, hac, logp, hba, hbd,
                       rotb, fsp3, psa, qed, pains, brenk, nih, zinc,
                       inchikey, tranche, reagent_a, reagent_b,
                       tanimoto_sml(mfp, morganbv_fp(%s::mol)) AS similarity
                  FROM enum_library
                 WHERE {where_clause}
                   AND tanimoto_sml(mfp, morganbv_fp(%s::mol)) >= %s
                 ORDER BY mfp <%%> morganbv_fp(%s::mol)
                 LIMIT %s;
            """
            full_params = [query_smiles] + params + \
                          [query_smiles, sim_thresh, query_smiles, limit]
        else:
            sql = f"""
                SELECT smiles, code, reaction, lab, mw, hac, logp, hba, hbd,
                       rotb, fsp3, psa, qed, pains, brenk, nih, zinc,
                       inchikey, tranche, reagent_a, reagent_b,
                       NULL::REAL AS similarity
                  FROM enum_library
                 WHERE {where_clause}
                 ORDER BY qed DESC NULLS LAST
                 LIMIT %s;
            """
            full_params = params + [limit]

        with psycopg2.connect(**self.conn_params) as conn:
            import warnings
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore",
                    message=".*pandas only supports SQLAlchemy.*",
                    category=UserWarning)
                return pd.read_sql(sql, conn, params=full_params)

    # ------------------------------------------------------------------
    def estimate_matches(self, **kwargs):
        """Fast (sub-second) row-count *estimate* via the planner.

        We deliberately do NOT run COUNT(*) here -- on this dataset that's a
        70+ second sequential scan in the worst case. Instead, we ask
        PostgreSQL for the planner's row estimate via EXPLAIN, which is
        instant and accurate to within a factor of ~2 once stats are fresh.
        """
        kwargs = dict(kwargs)
        kwargs.pop("query_smiles", None)
        kwargs.pop("sim_thresh", None)
        kwargs.pop("limit", None)

        where = []
        params = []
        if kwargs.get("labs"):
            labs = kwargs["labs"]
            if len(labs) == 1:
                where.append("lab = %s"); params.append(labs[0])
            else:
                where.append("lab = ANY(%s)"); params.append(list(labs))
        if kwargs.get("reactions"):
            rxns = kwargs["reactions"]
            if len(rxns) == 1:
                where.append("reaction = %s"); params.append(rxns[0])
            else:
                where.append("reaction = ANY(%s)"); params.append(list(rxns))
        for col in ["mw","hba","hbd","logp","rotb","psa","fsp3","qed"]:
            lo, hi = kwargs[col]
            where.append(f"{col} BETWEEN %s AND %s"); params.extend([lo, hi])
        if kwargs.get("exclude_pains"): where.append("(pains IS NULL OR pains = 0)")
        if kwargs.get("exclude_brenk"): where.append("(brenk IS NULL OR brenk = 0)")
        if kwargs.get("exclude_nih"):   where.append("(nih   IS NULL OR nih   = 0)")
        if kwargs.get("in_zinc"):       where.append("zinc = 1")

        where_clause = " AND ".join(where) if where else "TRUE"
        sql = f"EXPLAIN (FORMAT JSON) SELECT 1 FROM enum_library WHERE {where_clause};"

        with psycopg2.connect(**self.conn_params) as conn:
            with conn.cursor() as cur:
                cur.execute(sql, params)
                plan = cur.fetchone()[0][0]["Plan"]
                # Walk down to the leaf node which has the row estimate
                while "Plans" in plan:
                    plan = plan["Plans"][0]
                return int(plan.get("Plan Rows", 0))


print("✓ EnumLibraryDB ready!")


In [ ]:

# ============================================================================
# CELL 3: GUI
# ============================================================================
#@title Build the GUI

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from rdkit import Chem
from rdkit.Chem import Draw

db = EnumLibraryDB()

# ----- styling helpers -----------------------------------------------------
DESC_STYLE = {"description_width": "180px"}
SLIDER_LAYOUT = widgets.Layout(width="500px")
TOGGLE_LAYOUT = widgets.Layout(width="auto", margin="0 0 4px 0")
TAB_BOX_LAYOUT = widgets.Layout(border="1px solid #ccc", padding="10px",
                                margin="0 0 10px 0")

# ===========================================================================
# Reaction selection — one set of toggles per "scope" (Search All + each lab)
# ===========================================================================
# Each scope has its own dict of toggles so users can keep different
# selections per tab.  When a search runs we read the toggles of whichever
# tab is currently active.

scope_toggles = {}   # scope_name -> {reaction_code: ToggleButton}

def make_reaction_toggle(rxn):
    label = f"{rxn}: {REACTION_LABELS.get(rxn, rxn)}"
    return widgets.ToggleButton(value=False, description=label,
                                layout=TOGGLE_LAYOUT)

def build_search_all_panel():
    """All reactions grouped by lab, all selectable."""
    toggles = {}
    columns = []
    for lab, rxns in LAB_REACTIONS.items():
        title = widgets.HTML(f"<h4 style='margin:4px 0 6px 0'>{lab} Lab</h4>")
        col_widgets = [title]
        for r in rxns:
            t = make_reaction_toggle(r)
            toggles[r] = t
            col_widgets.append(t)
        columns.append(widgets.VBox(col_widgets,
                       layout=widgets.Layout(margin="0 20px 0 0")))
    scope_toggles["__all__"] = toggles
    select_all = widgets.Button(description="Select all", layout=widgets.Layout(width="100px"))
    deselect_all = widgets.Button(description="Deselect all", layout=widgets.Layout(width="120px"))
    def _sa(b):
        for t in toggles.values(): t.value = True
    def _da(b):
        for t in toggles.values(): t.value = False
    select_all.on_click(_sa); deselect_all.on_click(_da)
    bar = widgets.HBox([select_all, deselect_all],
                       layout=widgets.Layout(margin="0 0 8px 0"))
    return widgets.VBox([
        widgets.HTML("<i>Pick any combination of reactions across all labs.</i>"),
        bar,
        widgets.HBox(columns, layout=widgets.Layout(flex_flow="row wrap")),
    ], layout=TAB_BOX_LAYOUT)

def build_lab_panel(lab, reactions):
    """Just one lab's reactions."""
    toggles = {}
    rows = [widgets.HTML(f"<h4 style='margin:4px 0 6px 0'>{lab} Lab</h4>")]
    for r in reactions:
        t = make_reaction_toggle(r)
        toggles[r] = t
        rows.append(t)
    scope_toggles[lab] = toggles
    return widgets.VBox(rows, layout=TAB_BOX_LAYOUT)

# Build the tab widget
tab_titles = ["Search all"] + list(LAB_REACTIONS.keys())
tab_panels = [build_search_all_panel()] + \
             [build_lab_panel(lab, rxns) for lab, rxns in LAB_REACTIONS.items()]
reaction_tabs = widgets.Tab(children=tab_panels)
for i, t in enumerate(tab_titles):
    reaction_tabs.set_title(i, t)

def current_scope():
    return "__all__" if reaction_tabs.selected_index == 0 \
                     else tab_titles[reaction_tabs.selected_index]

# ===========================================================================
# Property filters (shared across all tabs)
# ===========================================================================
def rng(desc, lo, hi, step, val_lo, val_hi, fmt=".2f"):
    return widgets.FloatRangeSlider(
        value=[val_lo, val_hi], min=lo, max=hi, step=step,
        description=desc, style=DESC_STYLE, layout=SLIDER_LAYOUT,
        continuous_update=False, readout_format=fmt,
    )

n_results_slider = widgets.IntSlider(
    value=100, min=10, max=2000, step=10,
    description="Number of compounds:", style=DESC_STYLE, layout=SLIDER_LAYOUT,
    continuous_update=False)
mw_r    = rng("Molecular weight:",     100, 800, 10, 300, 450, ".0f")
hba_r   = rng("Hydrogen bond acceptors:", 0, 20, 1,  0, 10, ".0f")
hbd_r   = rng("Hydrogen bond donors:",    0, 10, 1,  0,  5, ".0f")
logp_r  = rng("cLogP:",                  -5,  8, 0.1, -3.0, 5.0)
rotb_r  = rng("Rotatable bonds:",         0, 20, 1,  0, 10, ".0f")
psa_r   = rng("TPSA:",                    0, 250, 1,  0, 140, ".0f")
fsp3_r  = rng("Fsp3:",                  0.0, 1.0, 0.05, 0.0, 1.0)
qed_r   = rng("QED:",                   0.0, 1.0, 0.05, 0.5, 1.0)

filters_box = widgets.VBox([
    n_results_slider, mw_r, hba_r, hbd_r, logp_r, rotb_r, psa_r, fsp3_r, qed_r
], layout=TAB_BOX_LAYOUT)

# Alert filters
exclude_pains = widgets.Checkbox(value=True,  description="Exclude PAINS")
exclude_brenk = widgets.Checkbox(value=False, description="Exclude Brenk")
exclude_nih   = widgets.Checkbox(value=False, description="Exclude NIH")
in_zinc       = widgets.Checkbox(value=False, description="In ZINC only")
alerts_box = widgets.HBox(
    [exclude_pains, exclude_brenk, exclude_nih, in_zinc],
    layout=TAB_BOX_LAYOUT)

# Similarity search
sim_query = widgets.Text(
    value="", description="Query SMILES (optional):",
    placeholder="leave blank to skip similarity search",
    style=DESC_STYLE, layout=widgets.Layout(width="700px"))
sim_thresh = widgets.FloatSlider(
    value=0.3, min=0.0, max=1.0, step=0.05,
    description="Min similarity:", style=DESC_STYLE, layout=SLIDER_LAYOUT,
    continuous_update=False, readout_format=".2f")
similarity_box = widgets.VBox([sim_query, sim_thresh], layout=TAB_BOX_LAYOUT)

# Buttons
count_button    = widgets.Button(description="Estimate count", button_style="info")
search_button   = widgets.Button(description="Search", button_style="success")
reset_button    = widgets.Button(description="Reset filters")
download_button = widgets.Button(description="Download CSV", button_style="warning",
                                 disabled=True)
buttons_box = widgets.HBox([count_button, search_button, reset_button, download_button])
output_area = widgets.Output()

# ===========================================================================
# Search logic
# ===========================================================================
results_df = pd.DataFrame()

def gather_filters():
    scope = current_scope()
    selected_reactions = [r for r, t in scope_toggles[scope].items() if t.value]

    # If a single-lab tab is active and reactions are selected from it, also
    # constrain by lab. For "Search all", we constrain by reaction only.
    if scope == "__all__":
        selected_labs = []
    else:
        selected_labs = [scope] if selected_reactions else []

    return dict(
        labs=selected_labs,
        reactions=selected_reactions,
        mw=tuple(mw_r.value),
        hba=tuple(hba_r.value),
        hbd=tuple(hbd_r.value),
        logp=tuple(logp_r.value),
        rotb=tuple(rotb_r.value),
        psa=tuple(psa_r.value),
        fsp3=tuple(fsp3_r.value),
        qed=tuple(qed_r.value),
        exclude_pains=exclude_pains.value,
        exclude_brenk=exclude_brenk.value,
        exclude_nih=exclude_nih.value,
        in_zinc=in_zinc.value,
        query_smiles=(sim_query.value.strip() or None),
        sim_thresh=sim_thresh.value,
        limit=n_results_slider.value,
    )

def on_count(b):
    with output_area:
        clear_output()
        try:
            kwargs = gather_filters()
            scope = current_scope()
            print(f"Scope: {'All labs' if scope == '__all__' else scope + ' lab'}")
            if not kwargs["reactions"]:
                print("⚠ No reactions selected on this tab.")
                return
            print(f"Reactions: {', '.join(kwargs['reactions'])}")
            n = db.estimate_matches(**kwargs)
            print(f"\n≈ {n:,} compounds estimated to match (planner estimate, not exact).")
            print(f"  Click Search to see the top {kwargs['limit']} results.")
            if kwargs["query_smiles"]:
                print(f"  (Similarity ranking will further narrow these "
                      f"to your top {kwargs['limit']} most similar.)")
        except Exception as e:
            print(f"❌ {e}")

def on_search(b):
    global results_df

    # Disable buttons and show searching state
    search_button.description = "Searching..."
    search_button.disabled = True
    search_button.button_style = "warning"
    count_button.disabled = True
    download_button.disabled = True

    try:
        with output_area:
            clear_output()
            kwargs = gather_filters()
            if not kwargs["reactions"]:
                print("⚠ Please toggle at least one reaction on the active tab.")
                return
            if kwargs["query_smiles"]:
                mol = Chem.MolFromSmiles(kwargs["query_smiles"])
                if mol is None:
                    print("❌ Invalid query SMILES.")
                    return
            scope = current_scope()
            print(f"⏳ Searching... (scope: {'All' if scope=='__all__' else scope}, "
                  f"reactions: {', '.join(kwargs['reactions'])}, "
                  f"limit: {kwargs['limit']})")
            print(f"   First search after a restart can take 30-90s. Subsequent ones are fast.")
            import time; t0 = time.time()
            try:
                results_df = db.search(**kwargs)
            except Exception as e:
                print(f"❌ {e}")
                return
            print(f"✓ Got {len(results_df):,} compounds in {time.time()-t0:.1f}s\n")
            if results_df.empty:
                print("No matches. Try widening the filters.")
                return

            download_button.disabled = False
            cols = ["code", "lab", "reaction", "mw", "logp", "qed"]
            if results_df["similarity"].notna().any():
                cols.append("similarity")
            display(results_df[cols + ["smiles"]].head(20))

            n_show = min(12, len(results_df))
            mols, legends = [], []
            for _, row in results_df.head(n_show).iterrows():
                m = Chem.MolFromSmiles(row["smiles"])
                if not m: continue
                mols.append(m)
                leg = f"{row['code']}\n{row['lab']} | {row['reaction']}\nMW {row['mw']:.0f} | QED {row['qed']:.2f}"
                if pd.notna(row["similarity"]):
                    leg += f"\nsim {row['similarity']:.2f}"
                legends.append(leg)
            if mols:
                img = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(280,280),
                                           legends=legends, returnPNG=False)
                display(img)
    finally:
        # Re-enable buttons no matter what happened
        search_button.description = "Search"
        search_button.disabled = False
        search_button.button_style = "success"
        count_button.disabled = False

def on_reset(b):
    for toggles in scope_toggles.values():
        for tog in toggles.values():
            tog.value = False
    n_results_slider.value = 100
    mw_r.value, hba_r.value, hbd_r.value = (300,450), (0,10), (0,5)
    logp_r.value, rotb_r.value, psa_r.value = (-3,5), (0,10), (0,140)
    fsp3_r.value, qed_r.value = (0,1), (0.5,1)
    exclude_pains.value, exclude_brenk.value = True, False
    exclude_nih.value, in_zinc.value = False, False
    sim_query.value = ""
    sim_thresh.value = 0.3
    with output_area:
        clear_output()
        print("Filters reset.")

def on_download(b):
    if results_df.empty:
        return
    fname = "enum_library_results.csv"
    results_df.to_csv(fname, index=False)
    try:
        from google.colab import files
        files.download(fname)
        print(f"✓ Downloaded {fname}")
    except Exception:
        print(f"✓ Saved to {fname}")

count_button.on_click(on_count)
search_button.on_click(on_search)
reset_button.on_click(on_reset)
download_button.on_click(on_download)

# ===========================================================================
# Assemble the UI
# ===========================================================================
display(HTML("<h3>Enumerated Hit-Expansion Library</h3>"))
display(HTML("<p style='color:#666;font-size:0.9em'><i>Tip: the first search after a database restart "
             "may take 30-90s while the cache warms up. Subsequent searches are fast (under 1s).</i></p>"))
display(HTML("<b>Pick reactions:</b> "
             "<i>(use 'Search all' for cross-lab queries, or a lab tab for that lab only)</i>"))
display(reaction_tabs)
display(HTML("<b>Property filters:</b>"))
display(filters_box)
display(HTML("<b>Alerts:</b>"))
display(alerts_box)
display(HTML("<b>Similarity search (optional):</b>"))
display(similarity_box)
display(buttons_box)
display(output_area)
